# Lab 10: Introduction to APIs: Wikipedia

*INFO 4617/5617 · Week 10 · Chapter 10, [Introduction to APIs: Wikipedia](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-10-wikipedia.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After most steps, a **⏱ Microsprint** gives you three minutes, alone or with a neighbor, to change one thing and bring a result back to the class. Every step ends with a **💬 Discuss** box of questions to talk over before the next one; click it to open it.

By the end you'll have asked two of Wikipedia's APIs for one article's whole public record: its basic facts, every edit since 2003, its daily readers, who those readers are, and what the article connects to. Then you'll do the same with a client library in three lines.

**The article:** [University of Colorado Boulder](https://en.wikipedia.org/wiki/University_of_Colorado_Boulder), the chapter's case. You'll choose your own article for the chapter's Recommended Exercises, due Sunday; this lab builds the tools they need.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Step 9 uses mwclient, which `webdata` doesn't include; Step 9 says how to install it.

On Friday, run every cell above where we stopped (**Run → Run All Above Selected Cell**).

## Step 0 — Set up (5 minutes)

Import `requests`, `pandas`, and `matplotlib`. Then define the User-Agent every request will send. Wikimedia's [User-Agent policy](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy) asks every client to say who it is and how to reach its operator, in this form: `<client name>/<version> (<contact information>) <library name>/<version>`. Keep the first two parts in `CLIENT`, because Step 9's library adds its own name, and add `requests` and its version for `HEADERS`. A client the [rate limits](https://www.mediawiki.org/wiki/Wikimedia_APIs/Rate_limits) can identify only by its network address gets 10 requests a minute, shared with everyone on that network, which is everyone in this room. One with a User-Agent like this gets 200, and if it still gets a 429, the response's `Retry-After` header says how many seconds to wait.

In [ ]:
# Import requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define CLIENT as "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Define HEADERS: a dict whose "User-Agent" is CLIENT, a space, then "python-requests/" and requests.__version__
# Define API, the Action API's address, and ARTICLE, the article's title
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 2 we wrote a User-Agent too. What does this one add, and who reads it?
- Why does the rate limit treat everyone in this room as one client, unless each of us says who we are?
- Wikipedia's `robots.txt` (week 2) disallows `/w/`, yet the API lives at `/w/api.php`. Why isn't that a contradiction?

</details>

## Step 1 — One request (5 minutes)

Every request to the MediaWiki [Action API](https://www.mediawiki.org/wiki/API:Action_API) has four parts: the base URL, an `action` (what to do), parameters that narrow it, and a `format`. Build the parameters as a dictionary, and `requests` turns them into the URL. Ask for [`prop=info`](https://www.mediawiki.org/wiki/API:Info), the article's basic facts, and look at the status code before anything else.

In [ ]:
# Build params: action=query, titles=ARTICLE, prop=info, format=json
# Send the request with HEADERS, print the status code, stop if it isn't 200, then print the JSON
# Your code here

<details>
<summary>💬 Discuss</summary>

- Our first request in week 1 fetched this same article as HTML. What's different about what came back this time?
- Print `response.url`. Where are the four parts of the request in it (week 5's URL anatomy), and which one would you change to ask about another article?
- What would a 403 or a 429 here tell you, and which of them is about you?

</details>

## Step 2 — Read the JSON one level at a time (7 minutes)

The answer is nested: `data["query"]["pages"]`, then a key that is the page's ID. We don't know the ID in advance, so take the first (and only) value. Print the title, the page ID, and the article's length in bytes.

In [ ]:
# Go down one level at a time: data -> "query" -> "pages" -> the one page inside
# Print its title, pageid, and length
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** ask `prop=info` about three titles in one request, joined with `|`: the article, its talk page (`Talk:` and the title), and a misspelling of the title. Print each page's key, title, and length. This is the chapter's tour of the talk page, and its warning about missing pages in "Common Issues to Debug".

**Share:** the talk page's length, and what the API did with the misspelling.

In [ ]:
# Microsprint: one request about three titles, joined with "|": ARTICLE, "Talk:" + ARTICLE, and a misspelling
# Send it as in Step 1; then for each key and page under ["query"]["pages"], print the key, the title,
#   and the length, or "missing" when there isn't one
# Your code here

<details>
<summary>💬 Discuss</summary>

- Why is the page ID a key rather than a value? What key did the misspelled title get, and what would `page["pageid"]` do with it?
- In week 6 we scraped Wikipedia infoboxes with BeautifulSoup. Which approach breaks first when Wikipedia redesigns its pages?
- The talk page is short, though editors have argued on it for years. Where did its discussions go, and what would you need to know before comparing two pages by length?

</details>

## Step 3 — Every edit, one batch at a time (15 minutes)

[`prop=revisions`](https://www.mediawiki.org/wiki/API:Revisions) returns an article's edits, newest first, at most 500 per request. When there are more, the response carries a [`"continue"`](https://www.mediawiki.org/wiki/API:Continue) key: add its contents to your parameters and ask again, until it's gone. Forgetting this loop is the most common API bug there is, because the first batch looks like a complete answer.

This step sends many requests, so first write `get(url, params)`, the one way every request from here on goes out. It always sends `HEADERS`, and when the answer is a 429 it waits as many seconds as `Retry-After` says and asks again, up to three times, as the rate-limit page asks.

Then write `get_revisions(title)`. Use `rvstart` to fix the newest edit you'll accept, `2026-09-30T23:59:59Z`, so that running it next week gives today's answer. Call [`raise_for_status()`](https://requests.readthedocs.io/en/latest/api/#requests.Response.raise_for_status) before `.json()`: if the rate limit still answers, you'll see why instead of a confusing JSON error.

In [ ]:
# Import time. Define get(url, params=None):
#   up to four tries: request url with params and HEADERS (timeout=30); return the response
#   unless it is a 429, in which case time.sleep() for int(response.headers["Retry-After"]) seconds
# Define get_revisions(title):
#   params: action=query, titles, prop=revisions, rvprop=ids|timestamp|user|size|comment,
#           rvlimit=max, rvstart="2026-09-30T23:59:59Z", format=json
#   loop: get(API, params), raise_for_status, keep this batch's revisions,
#         and if "continue" is in the response, add it to params; otherwise stop
#   return the revisions as a DataFrame
# Call it for ARTICLE and print how many revisions came back
# Your code here

<details>
<summary>💬 Discuss</summary>

- How many requests did `get_revisions()` send? Without the `continue` loop, how many revisions would we have, and would anything warn us?
- Why fix `rvstart` at a date in the past? What does that give someone who reruns your notebook next year?
- Week 7's Wayback captures and this history are both versions of a page over time. Which is the more complete record, and why?

</details>

## Step 4 — Chart the edits, and find the editors (7 minutes)

Turn `timestamp` into dates, count edits per month with [`resample("ME")`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html), and plot them, with a title and a label on each axis. Then use [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) on `user` to see who edits most, and what share of all edits the top ten made. The chapter's "Analyzing Revision History" does the same.

In [ ]:
# Convert timestamp to datetimes, count edits per month, and plot them with a title and axis labels
# Show the ten most active editors, and the share of all edits they made
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** count the edits made without an account. The API marks an edit from an IP address with a value in the `anon` column, and one from a temporary account with a value in `temp`; [`notna()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) finds them. What share of all edits are they, and what share of the edits since 2021? Count them, and don't print them: an IP address can point to a household. A short version of Additional Exercise 6's question of whether anonymous editors count.

**Share:** the two shares.

In [ ]:
# Microsprint: an edit without an account has a value in "anon" (an IP address) or "temp" (a temporary account)
# Count them without printing them: what share of all edits, and of the edits since 2021?
# Your code here

<details>
<summary>💬 Discuss</summary>

- Most edits came in the article's first few years. What does that suggest about how an article matures?
- In the microsprint, many edits came without an account, and each IP address counts as its own editor. What does that do to a count of editors, or to the share the top ten made?
- Usernames and IP addresses are both public here. By week 2's privacy question, is it fine to publish a ranking of an article's editors, or a list of its IP addresses?

</details>

## Step 5 — A second API for readers (5 minutes)

Pageviews come from a different API, Wikimedia's [REST API](https://wikimedia.org/api/rest_v1/), whose parameters are parts of the URL's path rather than a dictionary: project, access, **agent** (who is reading), article, granularity, start, and end. Define `get_pageviews()` in this cell, and use it in the next. Titles in the path use underscores, not spaces.

In [ ]:
# Define get_pageviews(title, start, end, agent="user", granularity="daily"):
#   build the URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article
#                  /en.wikipedia/all-access/{agent}/{article}/{granularity}/{start}/{end}
#   request it with get(), raise_for_status, make a DataFrame from ["items"]
#   turn "timestamp" (YYYYMMDDHH) into dates, and return the timestamp and views columns
# Your code here

<details>
<summary>💬 Discuss</summary>

- Week 5 called this a path-segment API, and the Action API a query-parameter one. Why would a title with a slash, such as AC/DC, break `get_pageviews()` but not Step 1's request?
- Why doesn't this function need a `continue` loop?
- In week 1 we asked this API for `all-agents`. What changes when we ask for `user`?

</details>

## Step 6 — Plot the readers, and find the biggest day (8 minutes)

Get daily views from January 1, 2021 to September 30, 2026, plot them with a title and axis labels, and find the peak with [`idxmax()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html). What happened that day? The API's days are UTC days, so an evening event in Colorado lands on the next one.

In [ ]:
# Get daily views for ARTICLE from 20210101 to 20260930, and plot them with a title and axis labels
# Find the day with the most views, and print its date and count
# Your code here

<details>
<summary>💬 Discuss</summary>

- The API's days are UTC days. What mistake could you make reporting the peak as a Colorado date?
- What happened around the peak? How would you check it with data, such as the revisions around that date, rather than from memory?
- A spike measures attention, not approval. Which claims can it support, and which can't it?

</details>

## Step 7 — Who is reading? (9 minutes)

A pageview is not a person. Wikimedia's [definition](https://meta.wikimedia.org/wiki/Research:Page_view) splits its counts by agent: `user` (people), `spider` (crawlers that say what they are), and `automated` (bots that don't). Call `get_pageviews()` once for each, with monthly granularity, for 2024 to September 2026, and plot the three together. When Wikimedia improved its bot detection in 2025, it moved past traffic from one agent to another. The chapter's "What a Pageview Counts" has the history.

In [ ]:
# For each agent in user, spider, automated: get monthly views from 20240101 to 20260930
# Put them in one DataFrame, one column per agent (pivot), and plot it with a title and axis labels
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** check the step's three lines. Ask for `agent="all-agents"`, monthly, for the same months, and check that each month is the sum of `user`, `spider`, and `automated`. Then divide `user` by it, and find the month when people were the smallest share of the views.

**Share:** whether the sum checks out, and that month and its share.

In [ ]:
# Microsprint: get monthly all-agents views for the same months; check each equals user + spider + automated
# Then divide user by all-agents, and print the month when people were the smallest share, with the share
# Your code here

<details>
<summary>💬 Discuss</summary>

- The microsprint found the month when people were the smallest share of views. How would a study that used `all-agents` have read that month?
- Wikimedia moved past traffic from one agent to another when its bot detection improved. What does that mean for anyone who downloaded counts before then?
- Our Selenium visits in week 8 came from a real browser. Which agent should they count as, and which do you think they did?

</details>

## Step 8 — One function for everything else (12 minutes)

Links, categories, templates, languages, and what links here are all [`action=query`](https://www.mediawiki.org/wiki/API:Query), with a different `prop` or `list`, and they all paginate the same way. So write the loop once: `query_all(params)` follows `"continue"` and returns every batch. Then count the article's categories, its language editions, and the articles that link to it.

Look at the first few categories: some are **hidden** maintenance categories that readers never see. Add `clshow="!hidden"` to leave them out.

In [ ]:
# Define query_all(params): add action=query and format=json, then loop like Step 3,
#   collecting each response's "query" part; return the list of batches
# Use it to count: categories (prop=categories, cllimit=max), with and without clshow="!hidden";
#   language editions (prop=langlinks, lllimit=max);
#   articles linking here (list=backlinks, bltitle, bllimit=max, blnamespace=0)
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** find which other titles send readers to this article. [`prop=redirects`](https://www.mediawiki.org/wiki/API:Redirects) isn't in the chapter: read its documentation for the name of its limit parameter, and add `rdnamespace=0` for articles only. Use `page_items()`, and print how many there are and the first eight. A short version of Additional Exercise 4.

**Share:** how many there are, and the most surprising one.

In [ ]:
# Microsprint: which titles redirect here? Use page_items() with prop="redirects",
#   its limit parameter (from the documentation) set to "max", and rdnamespace=0
# Print how many there are, and the first eight titles
# Your code here

<details>
<summary>💬 Discuss</summary>

- Why did `query_all()` work for categories, language links, backlinks, and the microsprint's redirects without a change?
- Hidden categories are maintenance labels that readers never see. Should a study of how Wikipedia classifies a topic include them?
- Dozens of other editions have an article on this topic. What could you learn by comparing them, as week 6 did with French infoboxes?

</details>

## Step 9 — The same with a client library (7 minutes)

[mwclient](https://mwclient.readthedocs.io/) does the continuation for you. If you haven't installed it, run `%conda install -y -c conda-forge mwclient` in a new cell first; conda-forge has version 0.10.1. Ask it for the same article's categories, language editions, and revisions. In mwclient 0.10, `limit` is how many results to ask for in each request; version 0.11 calls it `api_chunk_size`.

Two things stay raw. mwclient speaks only the Action API, so pageviews still need Step 5. And it doesn't wait on a 429: it stops with an error. So make the `Site` with `do_init=False`, which sends nothing yet, and give its session (`site.connection`) a retry rule from `urllib3`, the library under `requests`: on a 429, wait as long as `Retry-After` says and try again, up to five times. Then `site.site_init()` sends the first request.

In [ ]:
# Import mwclient and print its version
# Make a Site for en.wikipedia.org with clients_useragent=CLIENT (mwclient adds its own name and version)
#   and do_init=False; mount HTTPAdapter(max_retries=Retry(total=5, status_forcelist=[429]))
#   on site.connection for "https://", then call site.site_init()
# Get the page, then count its categories, its language editions,
#   and its revisions up to 2026-09-30T23:59:59Z, 500 per request (limit=500)
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** ask mwclient for the edits that the chapter's "Getting User Contributions" got with `list=usercontribs`: Jimbo Wales's, from January to September 2026. Use [`site.usercontributions()`](https://mwclient.readthedocs.io/en/latest/reference/site.html), whose `start` is the later date, because it goes newest first. Count the edits, then count them by `ns`, the [namespace](https://www.mediawiki.org/wiki/Help:Namespaces): 0 is articles, 1 their talk pages, 3 user talk pages. This is the last part of Additional Exercise 4: the same request with mwclient.

**Share:** the total (does it match the chapter's?), and how many were edits to articles.

In [ ]:
# Microsprint: list site.usercontributions("Jimbo Wales", start="2026-09-30T23:59:59Z",
#   end="2026-01-01T00:00:00Z", limit=500) into a DataFrame
# Print how many edits, and count them by "ns" with value_counts()
# Your code here

<details>
<summary>💬 Discuss</summary>

- What did mwclient do for you that you wrote by hand in Steps 3 and 8? What didn't it do?
- Most of those edits were on talk pages, not articles. What would a study that counted "edits" without splitting them by namespace get wrong?
- Week 3 was about APIs that closed. Wikipedia's has stayed open. What keeps it open, and what would your project lose if it closed?

</details>

## Step 10 — What did we learn? (5 minutes)

In a few sentences, as comments:

- Steps 3 and 9 counted the revisions two ways. Did they agree, and what would it mean if they didn't?
- In Step 7 and its microsprint, how much of the article's traffic was people? What would a study that used `all-agents` have gotten wrong?
- Which of today's numbers would be different if you ran this notebook next year, and which wouldn't? Why?
- What did mwclient do for you in Step 9 and its microsprint, and what did you still do yourself?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, choose an article and get its pageviews: Steps 5–6 built and used `get_pageviews()`, with a window that ends before today.
- Step 2, find the spike: Step 6's `idxmax()` and its warning about UTC days; Step 3's revisions are where to look around the date.
- Step 3, get the revision history: Step 3's `get()` and `get_revisions()`, with the `continue` loop.
- Step 4, profile the editors: Step 4's `value_counts()`, and its microsprint on edits without an account.
- Step 5, align edits with attention: Step 4's `resample("ME")`, and Step 7's monthly granularity.
- Step 6, map the neighborhood: Step 8's `query_all()` and `page_items()`, which the chapter calls `page_list()`; wikilinks are `prop=links`, with `pllimit`.
- Step 7, interpret: the Discuss questions after Steps 4, 6, and 7.

In [ ]:
# Your answers here, as comments


## Friday: from the lab to the textbook (15 minutes)

The last 15 minutes go to the textbook. Look back at what the lab turned up in [chapter 10](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-10-wikipedia.html):

- **An output that has changed.** The chapter printed its outputs in October 2026. Compare a few with yours, such as the article's length (Step 2) and the number of articles that link to it (Step 8). Which of them should the chapter fix to a date, and how?
- **A cell that broke.** Did a 429 stop one of your cells? The chapter's functions send `requests.get()` directly, and its mwclient code has no retry rule. Where should the chapter show how to wait?
- **A sentence that confused.** "Common Issues to Debug" says that a missing page's ID will be negative. What did Step 2's microsprint show?
- **A microsprint that would make a good exercise.** Checking that `all-agents` is the sum of the other three, or counting the edits made without an account, could be an Additional Exercise.

The [revision framework](https://github.com/cuinfoscience/INFO4617-Fall2026/blob/main/handouts/common/revision-framework.md) says how to propose a change, and what each kind needs to be accepted.